In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / "data" / "db" / "retail.db"

TABLE_NAMES = ["stores", "products", "customers", "orders", "order_items", "transactions"]

conn = sqlite3.connect(DB_PATH)
tables = {name: pd.read_sql(f"SELECT * FROM raw_{name}", conn) for name in TABLE_NAMES}

for name, df in tables.items():
    print(f"{name:<14} {df.shape}")

stores         (12, 8)
products       (300, 8)
customers      (4015, 7)
orders         (82089, 8)
order_items    (204230, 9)
transactions   (84390, 8)


In [2]:
AUDIT_COLUMNS = ["_source_file", "_loaded_at"]

blank_rows = []
for name, df in tables.items():
    data = df.drop(columns=AUDIT_COLUMNS)
    blank_counts = data.apply(lambda col: col.str.strip() == "").sum()

    for column, count in blank_counts.items():
        if count > 0:
            blank_rows.append({
                "table": name,
                "column": column,
                "blank_count": count,
                "blank_pct": round(100 * count / len(df), 2),
            })

blanks = pd.DataFrame(blank_rows)
blanks

,table,column,blank_count,blank_pct
0,customers,city,50,1.25
1,orders,customer_id,246,0.30
2,orders,order_ts,7,0.01
3,order_items,unit_price,40,0.02


In [3]:
PRIMARY_KEYS = {
    "stores": "store_id",
    "products": "product_id",
    "customers": "customer_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "transactions": "transaction_id",
}

duplicate_rows = []
for name, df in tables.items():
    data = df.drop(columns=AUDIT_COLUMNS)
    key = PRIMARY_KEYS[name]

    exact = data.duplicated().sum()
    same_key = data[key].duplicated().sum()

    duplicate_rows.append({
        "table": name,
        "primary_key": key,
        "exact_duplicate_rows": exact,
        "duplicate_ids": same_key,
        "same_id_different_content": same_key - exact,
    })

duplicates = pd.DataFrame(duplicate_rows)
duplicates

,table,primary_key,exact_duplicate_rows,duplicate_ids,same_id_different_content
0,stores,store_id,0,0,0
1,products,product_id,0,0,0
2,customers,customer_id,15,15,0
3,orders,order_id,326,327,1
4,order_items,order_item_id,0,0,0
5,transactions,transaction_id,0,0,0


In [4]:
orders = tables["orders"].drop(columns=AUDIT_COLUMNS)

repeated = orders[orders["order_id"].duplicated(keep=False)]
differs = repeated.groupby("order_id").nunique()
conflict_ids = differs[(differs > 1).any(axis=1)].index

orders[orders["order_id"].isin(conflict_ids)]

,order_id,customer_id,store_id,order_ts,channel,status
14049,O0014050,,S004,2024-09-25 12:10:00,Online,Completed
81895,O0014050,C02866,S004,2024-09-25 12:10:00,Online,Completed


In [5]:
DATE_COLUMNS = {
    ("orders", "order_ts"): "%Y-%m-%d %H:%M:%S",
    ("transactions", "transaction_ts"): "%Y-%m-%d %H:%M:%S",
    ("customers", "signup_date"): "%Y-%m-%d",
    ("stores", "opened_date"): "%Y-%m-%d",
}

TODAY = pd.Timestamp.today().normalize()

date_rows = []
for (table, column), date_format in DATE_COLUMNS.items():
    raw = tables[table][column].str.strip()
    parsed = pd.to_datetime(raw, format=date_format, errors="coerce")

    date_rows.append({
        "table": table,
        "column": column,
        "blank": (raw == "").sum(),
        "unparseable": (parsed.isna() & (raw != "")).sum(),
        "in_future": (parsed > TODAY).sum(),
        "earliest": parsed.min(),
        "latest_valid": parsed[parsed <= TODAY].max(),
    })

date_checks = pd.DataFrame(date_rows)
date_checks

,table,column,blank,unparseable,in_future,earliest,latest_valid
0,orders,order_ts,7,8,10,2024-01-02 16:34:00,2025-12-31 22:32:00
1,transactions,transaction_ts,0,0,0,2024-01-02 16:36:31,2025-12-31 22:34:20
2,customers,signup_date,0,0,0,2024-01-01 00:00:00,2025-12-23 00:00:00
3,stores,opened_date,0,0,0,2019-04-06 00:00:00,2022-08-24 00:00:00


In [6]:
order_dates = pd.to_datetime(tables["orders"]["order_ts"].str.strip(),
                             format="%Y-%m-%d %H:%M:%S", errors="coerce")
bad_date_orders = tables["orders"][order_dates.isna() | (order_dates > TODAY)]

payments = tables["transactions"]
successful = payments[payments["status"] == "Success"][["order_id", "transaction_ts"]]
successful = successful.drop_duplicates("order_id")

recovered = bad_date_orders[["order_id", "order_ts"]].merge(successful, on="order_id", how="left")

print("Orders with bad dates:", len(recovered))
print("Of those, with a valid payment time:", recovered["transaction_ts"].ne("").sum())
recovered.head(6)

Orders with bad dates: 25
Of those, with a valid payment time: 25


,order_id,order_ts,transaction_ts
0,O0002177,2099-01-01 10:00:00,2024-04-15 13:05:15
1,O0004650,31/02/2025,2024-05-31 21:40:49
2,O0004708,,2024-06-01 21:20:34
3,O0010964,31/02/2025,2024-08-25 14:48:44
4,O0015267,31/02/2025,2024-10-05 14:09:58
5,O0016312,31/02/2025,2024-10-12 10:53:27


In [7]:
NUMBER_RULES = {
    ("order_items", "quantity"): (1, None),
    ("order_items", "unit_price"): (0.01, None),
    ("order_items", "discount_pct"): (0, 100),
    ("products", "list_price"): (0.01, None),
    ("products", "unit_cost"): (0.01, None),
    ("transactions", "amount"): (0.01, None),
    ("customers", "birth_year"): (1920, 2010),
}

number_rows = []
for (table, column), (min_ok, max_ok) in NUMBER_RULES.items():
    raw = tables[table][column].str.strip()
    numbers = pd.to_numeric(raw, errors="coerce")

    number_rows.append({
        "table": table,
        "column": column,
        "blank": (raw == "").sum(),
        "not_a_number": (numbers.isna() & (raw != "")).sum(),
        "below_min": (numbers < min_ok).sum(),
        "above_max": (numbers > max_ok).sum() if max_ok is not None else 0,
        "min": numbers.min(),
        "median": numbers.median(),
        "max": numbers.max(),
    })

number_checks = pd.DataFrame(number_rows)
number_checks

,table,column,blank,not_a_number,below_min,above_max,min,median,max
0,order_items,quantity,0,0,60,0,-4.00,2.00,117.00
1,order_items,unit_price,40,0,0,0,40.00,1640.00,14640.00
2,order_items,discount_pct,0,0,0,0,0.00,5.00,30.00
3,products,list_price,0,0,0,0,40.00,935.00,14640.00
4,products,unit_cost,0,0,0,0,26.17,676.01,10131.69
5,transactions,amount,0,0,0,0,28.00,5588.00,1437940.00
6,customers,birth_year,0,0,0,0,1965.00,1985.00,2005.00


In [8]:
ALLOWED_VALUES = {
    ("products", "category"): ["Electronics", "Home & Kitchen", "Fashion", "Grocery", "Sports"],
    ("stores", "region"): ["North", "South", "East", "West"],
    ("stores", "store_type"): ["Flagship", "Standard", "Express"],
    ("customers", "segment"): ["Regular", "Occasional", "Premium"],
    ("orders", "channel"): ["In-store", "Online"],
    ("orders", "status"): ["Completed"],
    ("order_items", "is_returned"): ["True", "False"],
    ("transactions", "payment_method"): ["UPI", "Card", "Cash", "Wallet"],
    ("transactions", "status"): ["Success", "Failed"],
}

category_rows = []
for (table, column), allowed in ALLOWED_VALUES.items():
    values = tables[table][column]
    invalid = values[~values.isin(allowed)]

    category_rows.append({
        "table": table,
        "column": column,
        "distinct_values": values.nunique(),
        "invalid_rows": len(invalid),
        "invalid_examples": sorted(invalid.unique())[:6],
    })

category_checks = pd.DataFrame(category_rows)
category_checks

,table,column,distinct_values,invalid_rows,invalid_examples
0,products,category,16,36,"[Apparel, Electronics , FASHION , GROCERY, Hom..."
1,stores,region,4,0,[]
2,stores,store_type,3,0,[]
3,customers,segment,3,0,[]
4,orders,channel,2,0,[]
5,orders,status,1,0,[]
6,order_items,is_returned,2,0,[]
7,transactions,payment_method,4,0,[]
8,transactions,status,2,0,[]


In [9]:
tables["products"]["category"].map(repr).value_counts()

category
'Grocery'             71
'Fashion'             65
'Home & Kitchen'      52
'Electronics'         48
'Sports'              28
'grocery'              6
'fashion'              5
'FASHION '             4
'Apparel'              4
'GROCERY'              4
'Home&Kitchen'         3
'electronics'          3
'Electronics '         2
'home & kitchen'       2
'Home and Kitchen'     2
'sports'               1
Name: count, dtype: int64

In [10]:
LINKS = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("orders", "store_id", "stores", "store_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("transactions", "order_id", "orders", "order_id"),
]

link_rows = []
for child_table, child_col, parent_table, parent_col in LINKS:
    child_ids = tables[child_table][child_col]
    child_ids = child_ids[child_ids.str.strip() != ""]
    broken = ~child_ids.isin(tables[parent_table][parent_col])

    link_rows.append({
        "link": f"{child_table}.{child_col} -> {parent_table}.{parent_col}",
        "rows_checked": len(child_ids),
        "broken_links": broken.sum(),
    })

link_checks = pd.DataFrame(link_rows)
link_checks

,link,rows_checked,broken_links
0,orders.customer_id -> customers.customer_id,81843,0
1,orders.store_id -> stores.store_id,82089,0
2,order_items.order_id -> orders.order_id,204230,0
3,order_items.product_id -> products.product_id,204230,0
4,transactions.order_id -> orders.order_id,84390,0


In [11]:
orders = tables["orders"]
items = tables["order_items"]
payments = tables["transactions"]

no_items = ~orders["order_id"].isin(items["order_id"])
no_success_payment = ~orders["order_id"].isin(payments.loc[payments["status"] == "Success", "order_id"])

print("Orders with no items:", no_items.sum())
print("Orders with no successful payment:", no_success_payment.sum())

Orders with no items: 0
Orders with no successful payment: 0


In [12]:
findings = []

def add_finding(check, table, column, issue, rows_affected):
    if rows_affected > 0:
        findings.append({
            "check": check,
            "table": table,
            "column": column,
            "issue": issue,
            "rows_affected": int(rows_affected),
            "pct_of_table": round(100 * rows_affected / len(tables[table]), 2),
        })

for _, row in blanks.iterrows():
    add_finding("missing_values", row["table"], row["column"], "blank value", row["blank_count"])

for _, row in duplicates.iterrows():
    add_finding("duplicates", row["table"], row["primary_key"], "exact duplicate row", row["exact_duplicate_rows"])
    add_finding("duplicates", row["table"], row["primary_key"], "same ID, different content", row["same_id_different_content"])

for _, row in date_checks.iterrows():
    add_finding("dates", row["table"], row["column"], "unparseable date", row["unparseable"])
    add_finding("dates", row["table"], row["column"], "date in the future", row["in_future"])

for _, row in number_checks.iterrows():
    add_finding("numbers", row["table"], row["column"], "not a number", row["not_a_number"])
    add_finding("numbers", row["table"], row["column"], "below allowed minimum", row["below_min"])
    add_finding("numbers", row["table"], row["column"], "above allowed maximum", row["above_max"])

for _, row in category_checks.iterrows():
    add_finding("categories", row["table"], row["column"], "value not in allowed list", row["invalid_rows"])

for _, row in link_checks.iterrows():
    child_table = row["link"].split(".")[0]
    add_finding("links", child_table, row["link"], "ID not found in parent table", row["broken_links"])

report = pd.DataFrame(findings)
report

,check,table,column,issue,rows_affected,pct_of_table
0,missing_values,customers,city,blank value,50,1.25
1,missing_values,orders,customer_id,blank value,246,0.30
2,missing_values,orders,order_ts,blank value,7,0.01
3,missing_values,order_items,unit_price,blank value,40,0.02
4,duplicates,customers,customer_id,exact duplicate row,15,0.37
5,duplicates,orders,order_id,exact duplicate row,326,0.40
6,duplicates,orders,order_id,"same ID, different content",1,0.00
7,dates,orders,order_ts,unparseable date,8,0.01
8,dates,orders,order_ts,date in the future,10,0.01
9,numbers,order_items,quantity,below allowed minimum,60,0.03


In [13]:
DECISIONS = {
    ("customers", "city", "blank value"):
        ("Low", "Fill with 'Unknown'; city is not a model input"),
    ("orders", "customer_id", "blank value"):
        ("Medium", "Keep for sales forecasting; exclude from churn (can't be linked to a customer)"),
    ("orders", "order_ts", "blank value"):
        ("High", "Recover date from the successful payment time"),
    ("order_items", "unit_price", "blank value"):
        ("Medium", "Fill with the product's list price"),
    ("customers", "customer_id", "exact duplicate row"):
        ("Low", "Drop the extra copies"),
    ("orders", "order_id", "exact duplicate row"):
        ("High", "Drop the extra copies (otherwise sales are double counted)"),
    ("orders", "order_id", "same ID, different content"):
        ("High", "Keep the copy that has a customer_id"),
    ("orders", "order_ts", "unparseable date"):
        ("High", "Recover date from the successful payment time"),
    ("orders", "order_ts", "date in the future"):
        ("High", "Recover date from the successful payment time"),
    ("order_items", "quantity", "below allowed minimum"):
        ("High", "Remove these item lines (impossible purchases)"),
    ("products", "category", "value not in allowed list"):
        ("Medium", "Map variants to the 5 standard categories; assume 'Apparel' = 'Fashion'"),
}

NOT_DECIDED = ("Review", "Not decided yet")
report["severity"] = [DECISIONS.get((r.table, r.column, r.issue), NOT_DECIDED)[0] for r in report.itertuples()]
report["planned_action"] = [DECISIONS.get((r.table, r.column, r.issue), NOT_DECIDED)[1] for r in report.itertuples()]

report[["table", "column", "issue", "rows_affected", "severity", "planned_action"]]

,table,column,issue,rows_affected,severity,planned_action
0,customers,city,blank value,50,Low,Fill with 'Unknown'; city is not a model input
1,orders,customer_id,blank value,246,Medium,Keep for sales forecasting; exclude from churn...
2,orders,order_ts,blank value,7,High,Recover date from the successful payment time
3,order_items,unit_price,blank value,40,Medium,Fill with the product's list price
4,customers,customer_id,exact duplicate row,15,Low,Drop the extra copies
5,orders,order_id,exact duplicate row,326,High,Drop the extra copies (otherwise sales are dou...
6,orders,order_id,"same ID, different content",1,High,Keep the copy that has a customer_id
7,orders,order_ts,unparseable date,8,High,Recover date from the successful payment time
8,orders,order_ts,date in the future,10,High,Recover date from the successful payment time
9,order_items,quantity,below allowed minimum,60,High,Remove these item lines (impossible purchases)


In [14]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

report.to_csv(REPORTS_DIR / "data_quality_issues.csv", index=False)

passed = [
    "All 6 files have the expected columns (schema check, Step 1)",
    "All discounts are between 0 and 100%",
    "All dates in transactions, customers and stores are valid",
    "All IDs link to an existing parent record (5 links checked)",
    "Every order has at least one item and a successful payment",
]
noted = [
    "Some quantities (up to 117) and payment amounts (up to Rs 14.4 lakh) are far above normal. "
    "They break no rule, so they are not treated as data errors; they are investigated in Step 8 (anomaly detection).",
]

lines = ["# Data quality report", "", f"Generated: {pd.Timestamp.now():%Y-%m-%d %H:%M}", "", "## Rows per table", ""]
for name, df in tables.items():
    lines.append(f"- {name}: {len(df):,} rows")

lines += ["", "## Issues found", ""]
for r in report.itertuples():
    lines.append(f"- **[{r.severity}] {r.table}.{r.column}**: {r.issue}: "
                 f"{r.rows_affected:,} rows ({r.pct_of_table}%). Action: {r.planned_action}")

lines += ["", "## Checks that passed", ""] + [f"- {p}" for p in passed]
lines += ["", "## Noted, not treated as errors", ""] + [f"- {n}" for n in noted]

(REPORTS_DIR / "data_quality_report.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))

# Data quality report

Generated: 2026-10-26 15:59

## Rows per table

- stores: 12 rows
- products: 300 rows
- customers: 4,015 rows
- orders: 82,089 rows
- order_items: 204,230 rows
- transactions: 84,390 rows

## Issues found

- **[Low] customers.city**: blank value: 50 rows (1.25%). Action: Fill with 'Unknown'; city is not a model input
- **[Medium] orders.customer_id**: blank value: 246 rows (0.3%). Action: Keep for sales forecasting; exclude from churn (can't be linked to a customer)
- **[High] orders.order_ts**: blank value: 7 rows (0.01%). Action: Recover date from the successful payment time
- **[Medium] order_items.unit_price**: blank value: 40 rows (0.02%). Action: Fill with the product's list price
- **[Low] customers.customer_id**: exact duplicate row: 15 rows (0.37%). Action: Drop the extra copies
- **[High] orders.order_id**: exact duplicate row: 326 rows (0.4%). Action: Drop the extra copies (otherwise sales are double counted)
- **[High] orders.order_id**: same ID, dif

In [15]:
import json

answer_key_path = PROJECT_ROOT / "data" / "raw" / "_answer_key" / "planted_quality_issues.json"
planted = json.loads(answer_key_path.read_text())

def found(table, column, issue):
    match = report[(report["table"] == table) & (report["column"] == column) & (report["issue"] == issue)]
    return int(match["rows_affected"].sum())

our_counts = {
    "duplicate_order_rows": found("orders", "order_id", "exact duplicate row")
                            + found("orders", "order_id", "same ID, different content"),
    "missing_customer_id": found("orders", "customer_id", "blank value"),
    "invalid_order_dates": found("orders", "order_ts", "blank value")
                           + found("orders", "order_ts", "unparseable date")
                           + found("orders", "order_ts", "date in the future"),
    "impossible_quantities": found("order_items", "quantity", "below allowed minimum"),
    "missing_unit_price": found("order_items", "unit_price", "blank value"),
    "inconsistent_category_labels": found("products", "category", "value not in allowed list"),
    "duplicate_customer_rows": found("customers", "customer_id", "exact duplicate row"),
    "missing_customer_city": found("customers", "city", "blank value"),
}

comparison = pd.DataFrame({"planted": planted, "we_found": our_counts})
comparison["match"] = comparison["planted"] == comparison["we_found"]
comparison

,planted,we_found,match
duplicate_order_rows,327,327,True
missing_customer_id,246,246,True
invalid_order_dates,25,25,True
impossible_quantities,60,60,True
missing_unit_price,40,40,True
inconsistent_category_labels,36,36,True
duplicate_customer_rows,15,15,True
missing_customer_city,50,50,True
